In [ ]:
%cd /content/drive/MyDrive/TMA/NLPTraining
!ls

/content/drive/MyDrive/TMA/NLPTraining
GoogleNews-vectors-negative300.bin  Restaurant_Reviews.tsv


In [ ]:
!pip install Keras-Preprocessing

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.6/42.6 kB 1.8 MB/s eta 0:00:00


#Load data

In [ ]:
import pandas as pd
import numpy as np
# Import confusion_matrix and classification_report from the sklearn.metrics module
from sklearn.metrics import confusion_matrix, classification_report

pd.set_option('display.max_colwidth', None)
df = pd.read_csv('Restaurant_Reviews.tsv', sep='\t')
print(df)


                                                                                                                                     Review  \
0                                                                                                                  Wow... Loved this place.   
1                                                                                                                        Crust is not good.   
2                                                                                                 Not tasty and the texture was just nasty.   
3                                                   Stopped by during the late May bank holiday off Rick Steve recommendation and loved it.   
4                                                                               The selection on the menu was great and so were the prices.   
..                                                                                                                                      ...   

#Load model word2vec pretrained

#### Download link



> https://www.kaggle.com/datasets/leadbest/googlenewsvectorsnegative300



In [ ]:
import pandas as pd
from datetime import date, timedelta
import re
from nltk.tokenize import word_tokenize
from gensim import corpora, models
from gensim.models import KeyedVectors
from gensim.matutils import corpus2dense
import gensim

#Loading the word vectors from Google trained word2Vec model
GoogleModel = KeyedVectors.load_word2vec_format('GoogleNews-vectors-negative300.bin', binary=True)


#Test model

In [ ]:
word = 'apple'
vector = GoogleModel[word]
print(len(vector))

300


In [ ]:
similar_words = GoogleModel.most_similar(word)
print(similar_words)

[('apples', 0.720359742641449), ('pear', 0.6450697183609009), ('fruit', 0.6410146355628967), ('berry', 0.6302295327186584), ('pears', 0.613396167755127), ('strawberry', 0.6058260798454285), ('peach', 0.6025872826576233), ('potato', 0.5960935354232788), ('grape', 0.5935863852500916), ('blueberry', 0.5866668224334717)]


#Convert to list token of vocabulary

In [ ]:
from keras_preprocessing.text import Tokenizer
from keras_preprocessing import sequence

tokenizer = Tokenizer()
tokenizer.fit_on_texts(df['Review'])
sequences = tokenizer.texts_to_sequences(df['Review'])

maxlen = 100
X = sequence.pad_sequences(sequences, maxlen=maxlen)
# print('X:\n', X[:10])
print(X.shape)

(1000, 100)


#Create ember maxtrix

In [ ]:
import numpy as np

embedding_dim = 300
word_index = tokenizer.word_index
num_words = min(len(word_index) + 1, len(GoogleModel.index_to_key))
embedding_matrix = np.zeros((num_words, embedding_dim))

print('num_words:', num_words)
for word, i in word_index.items():
    if i >= num_words:
        continue
    if word in GoogleModel.index_to_key:
        embedding_matrix[i] = GoogleModel.word_vec(word)


num_words: 2072


<ipython-input-14-d7eb5b195804>:13: DeprecationWarning: Call to deprecated `word_vec` (Use get_vector instead).
  embedding_matrix[i] = GoogleModel.word_vec(word)


#Create model

In [ ]:
from keras.models import Sequential
from keras.layers import Embedding, Flatten, Dense, LSTM, Bidirectional

model = Sequential()
model.add(Embedding(num_words, embedding_dim, input_length=maxlen, weights=[embedding_matrix], trainable=False))
model.add(Bidirectional(LSTM(64, return_sequences=True, input_shape=(maxlen, ))))
model.add(Flatten())
model.add(Dense(1, activation='sigmoid'))

model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['acc'])
model.summary()


Model: "sequential"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 embedding (Embedding)       (None, 100, 300)          621600    
                                                                 
 bidirectional (Bidirection  (None, 100, 128)          186880    
 al)                                                             
                                                                 
 flatten (Flatten)           (None, 12800)             0         
                                                                 
 dense (Dense)               (None, 1)                 12801     
                                                                 
Total params: 821281 (3.13 MB)
Trainable params: 199681 (780.00 KB)
Non-trainable params: 621600 (2.37 MB)
_________________________________________________________________


#Create training and testing data

In [ ]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X, df['Liked'], test_size=0.3, random_state=101)


#Start Training

In [ ]:
model.fit(X_train, y_train, epochs=10, batch_size=32)

Epoch 1/10
22/22 [==============================] - 12s 191ms/step - loss: 0.6617 - acc: 0.6371
Epoch 2/10
22/22 [==============================] - 5s 243ms/step - loss: 0.4725 - acc: 0.7957
Epoch 3/10
22/22 [==============================] - 12s 546ms/step - loss: 0.3809 - acc: 0.8443
Epoch 4/10
22/22 [==============================] - 11s 487ms/step - loss: 0.3186 - acc: 0.8571
Epoch 5/10
22/22 [==============================] - 7s 291ms/step - loss: 0.2254 - acc: 0.9000
Epoch 6/10
22/22 [==============================] - 4s 179ms/step - loss: 0.1883 - acc: 0.9271
Epoch 7/10
22/22 [==============================] - 7s 347ms/step - loss: 0.2049 - acc: 0.9271
Epoch 8/10
22/22 [==============================] - 4s 176ms/step - loss: 0.1201 - acc: 0.9643
Epoch 9/10
22/22 [==============================] - 4s 180ms/step - loss: 0.0784 - acc: 0.9714
Epoch 10/10
22/22 [==============================] - 6s 270ms/step - loss: 0.0818 - acc: 0.9671


#Evaluation

In [ ]:
predictions = model.predict(X_test)
print('predict:', predictions[:3])
predictions = np.round(predictions)
print('predictions:', predictions.flatten())
print(confusion_matrix(y_test, predictions))
print(classification_report(y_test, predictions))

10/10 [==============================] - 2s 68ms/step
predict: [[0.9974591]
 [0.9784146]
 [0.8273906]]
predictions: [1. 1. 1. 0. 1. 1. 1. 1. 0. 1. 0. 0. 0. 1. 0. 1. 0. 1. 0. 1. 0. 0. 1. 1.
 0. 1. 1. 0. 1. 1. 1. 1. 1. 0. 1. 1. 0. 1. 0. 1. 1. 1. 1. 0. 0. 1. 0. 0.
 1. 0. 0. 0. 0. 0. 0. 1. 1. 0. 1. 0. 1. 0. 0. 1. 0. 0. 1. 1. 1. 0. 1. 0.
 0. 1. 1. 0. 0. 1. 1. 0. 0. 0. 1. 0. 0. 0. 1. 1. 1. 0. 0. 0. 0. 1. 1. 0.
 1. 1. 1. 1. 0. 0. 0. 1. 1. 0. 0. 0. 1. 0. 0. 1. 0. 0. 1. 1. 0. 1. 0. 0.
 0. 0. 1. 1. 0. 0. 0. 1. 1. 1. 1. 0. 1. 0. 1. 1. 1. 0. 1. 0. 0. 1. 0. 1.
 0. 0. 1. 0. 0. 0. 1. 1. 1. 0. 0. 0. 1. 0. 1. 0. 1. 1. 0. 1. 0. 1. 0. 1.
 1. 1. 1. 1. 0. 1. 0. 1. 1. 1. 0. 0. 1. 1. 0. 0. 0. 1. 0. 0. 0. 0. 0. 1.
 1. 0. 0. 0. 1. 1. 1. 1. 1. 0. 0. 1. 0. 0. 0. 1. 1. 1. 1. 0. 1. 1. 0. 0.
 1. 0. 1. 0. 1. 1. 0. 0. 0. 1. 0. 1. 0. 0. 0. 0. 0. 1. 1. 1. 0. 0. 1. 0.
 1. 1. 0. 0. 0. 0. 1. 0. 1. 0. 0. 1. 0. 0. 1. 1. 0. 1. 0. 1. 0. 0. 1. 0.
 1. 1. 0. 1. 1. 1. 1. 1. 1. 1. 0. 0. 1. 0. 1. 0. 1. 1. 0. 1. 0. 0. 1. 1.
 1. 1. 0